# bhavpy quickstart

Indian market data from NSE/BSE public archives. First run downloads one archive file per trading day for the range (whole market) into a local cache; after that everything is instant and offline.

*Not affiliated with NSE/BSE. Personal/research use; follow the exchanges' terms. Not investment advice.*

In [ ]:
# pip install "bhavpy[plot]"
import bhavpy as bp
import pandas as pd
import matplotlib.pyplot as plt
bp.__version__

## 1. Raw vs adjusted prices
RELIANCE had 1:1 bonuses in Sep 2017 and Oct 2024. The raw close halves on each ex-date; the adjusted close does not.

In [ ]:
df = bp.history("RELIANCE", "2017-01-01", "2025-12-31")
ax = df[["close", "adj_close"]].plot(figsize=(10, 4), title="RELIANCE: raw vs split/bonus-adjusted")

In [ ]:
bp.factors("RELIANCE", "2017-01-01", "2025-12-31", adjust="total")[
    ["ex_date", "kind", "ratio", "amount", "price_factor", "cumulative_factor", "check"]
].tail(10)

## 2. Old tickers still work
`ZOMATO` was renamed `ETERNAL` on 2025-04-09; the history is stitched.

In [ ]:
bp.symbol_chain("ZOMATO")

In [ ]:
bp.history("ZOMATO", "2025-04-01", "2025-04-15", adjust=None)[["symbol", "close"]]

## 3. Index history and point-in-time constituents

In [ ]:
nifty = bp.history("NIFTY 50", "2015-01-01", "2025-12-31")
nifty[["close", "pe"]].plot(subplots=True, figsize=(10, 5));

In [ ]:
then = set(bp.constituents("NIFTY 50", as_of="2018-01-01")["symbol"])
now = set(bp.constituents("NIFTY 50")["symbol"])
print("left since 2018:", sorted(then - now))
print("joined since 2018:", sorted(now - then))

## 4. Moving-average backtest on adjusted prices

In [ ]:
import numpy as np
px = bp.history("TCS", "2015-01-01", "2025-12-31", adjust="total")["adj_close"]
pos = (px.rolling(50).mean() > px.rolling(200).mean()).astype(int).shift(1).fillna(0)
ret = px.pct_change().fillna(0)
strat = pos * ret - 0.001 * pos.diff().abs().fillna(0)
pd.DataFrame({"buy & hold": (1 + ret).cumprod(), "SMA 50/200": (1 + strat).cumprod()}).plot(figsize=(10, 4), logy=True);

## 5. Calendar, F&O, flows

In [ ]:
bp.holidays(2026).head()

In [ ]:
bp.lot_sizes().query("symbol in ['NIFTY', 'BANKNIFTY']").head()

In [ ]:
bp.fii_dii()

## 6. Live option chain → IV smile
(The option chain is the one dataset read from NSE's live site.)

In [ ]:
chain = bp.option_chain("NIFTY")
spot = chain.attrs["underlying_price"]
otm = chain[((chain.option_type == "PE") & (chain.strike < spot)) | ((chain.option_type == "CE") & (chain.strike >= spot))]
otm = otm[(otm.iv > 0) & (otm.strike.between(spot * 0.9, spot * 1.1))]
ax = otm.plot(x="strike", y="iv", style="o-", title=f"NIFTY IV smile @ {chain.attrs['timestamp']}", figsize=(8, 4))
ax.axvline(spot, ls="--", c="grey");

In [ ]:
from bhavpy import cache
cache.info()["size_mb"], cache.path()